In [ ]:
%matplotlib widget

In [ ]:
import os
from pathlib import Path
from tbp.monty.frameworks.utils.logging_utils import load_stats
import matplotlib.pyplot as plt
from tbp.monty.frameworks.utils.plot_utils_dev import plot_graph
import matplotlib.colors as mcolors
import numpy as np
import pandas as pd
from tbp.monty.frameworks.utils.plot_utils import format_axes

In [ ]:
pretrain_path = os.path.expanduser("~/tbp/results/monty/pretrained_models/")
pretrained_dict = (
    pretrain_path
    + "pretrained_ycb_v13/surf_agent_1lm_77obj/pretrained/"
)

log_path = os.path.expanduser("~/tbp/results/monty/projects/evidence_eval_runs/logs/")
exp_name = "base_77obj_surf_agent_store_symmetry"

exp_path = Path(log_path) / exp_name

In [ ]:
file_name = "eval_stats_10epochs.csv"
eval_stats = pd.read_csv(exp_path / file_name)

In [ ]:
# Split pandas df into epochs (each epoch has 77 rows)
num_epochs = 10
# Filter eval_stats if object_subset is not None
object_subset = ["potted_meat_can", "master_chef_can", "i_cups", "spoon", "b_cups", "pitcher_base", "knife", "b_marbles", "h_cups", "strawberry", "power_drill", "padlock", "golf_ball", "hammer", "softball"]

if object_subset is not None:
    filtered_eval_stats = eval_stats[eval_stats["primary_target_object"].isin(object_subset)]
else:
    filtered_eval_stats = eval_stats

epochs = np.array_split(filtered_eval_stats, num_epochs)

avg_num_steps_list = []
avg_symmetry_evidence_list = []
percent_correct_list = []
percent_correct_mlh_list = []
avg_runtime_list = []

for i, epoch in enumerate(epochs):

    avg_num_steps = epoch["num_steps"].mean()
    avg_num_steps_list.append(avg_num_steps)

    avg_symmetry_evidence = epoch["symmetry_evidence"].mean()
    avg_symmetry_evidence_list.append(avg_symmetry_evidence)

    perf_counts = epoch["primary_performance"].value_counts()
    percent_correct = perf_counts.get("correct", 0) / len(epoch) * 100
    percent_correct_list.append(percent_correct)

    percent_correct_mlh = perf_counts.get("correct_mlh", 0) / len(epoch) * 100
    percent_correct_mlh_list.append(percent_correct_mlh)

    avg_runtime = epoch["time"].mean()
    avg_runtime_list.append(avg_runtime)


In [ ]:
plt.figure(figsize=(7, 7))
plt.subplot(2, 2, 1)
plt.plot(avg_num_steps_list)
ax1 = plt.gca()
ax2 = ax1.twinx()
ax2.plot(avg_runtime_list, color='orange')
ax2.set_ylabel('Average Runtime (s)', color='orange')
plt.title("Average Number of Steps")
plt.subplot(2, 2, 2)
plt.plot(avg_symmetry_evidence_list)
plt.title("Average Symmetry Evidence")
plt.subplot(2, 2, 3)
plt.plot(percent_correct_list)
plt.title("Percent Correct")
plt.subplot(2, 2, 4)
plt.plot(percent_correct_mlh_list)
plt.title("Percent Correct MLH")
plt.suptitle(f"Results for Objects: {object_subset}", fontsize=10)
plt.tight_layout()
plt.show()


In [ ]:
# Plot num_steps over epochs for each object

# Ensure eval_stats is available and contains 'primary_target_object' and 'num_steps' columns
objects = eval_stats['primary_target_object'].unique()

plt.figure(figsize=(8, 5))

for object_id in objects[:10]:
    # Extract num_steps for the current object in the order they appear (which goes over epochs)
    steps = eval_stats[eval_stats['primary_target_object'] == object_id]['num_steps'].values
    epochs = range(1, len(steps) + 1)  # Epochs as x-axis (1-indexed)
    if np.max(steps) <70:
        plt.plot(epochs, steps, marker='o', label=str(object_id))

plt.xlabel('Epoch')
plt.ylabel('Number of Steps')
plt.title('Number of Steps Over Epochs for Each Object')
plt.legend(title="Object")
plt.tight_layout()
plt.show()

In [ ]:
train_stats, eval_stats, detailed_stats, lm_models = load_stats(
    exp_path,
    load_train=False,  # doesn't load train csv
    load_eval=True,  # loads eval_stats.csv
    load_detailed=False,  # doesn't load .json
    load_models=True,  # loads .pt models
    pretrained_dict=pretrained_dict,
)

In [ ]:
lm_models

## Pretrained Models

In [ ]:
pretrain_path = os.path.expanduser("~/tbp/results/monty/pretrained_models/")
pretrained_dict_sym = (
    pretrain_path
    + "pretrained_ycb_v13/base_77obj_surf_agent_store_symmetry_marked_up/pretrained/"
)

log_path = os.path.expanduser("~/tbp/results/monty/projects/evidence_eval_runs/logs/")
exp_name_pretrained = "base_77obj_surf_agent_load_symmetry"

exp_path_pretrained = Path(log_path) / exp_name_pretrained

In [ ]:
_, eval_stats_pretrained, _, pretrained_lm_models = load_stats(
    exp_path_pretrained,
    load_train=False,  # doesn't load train csv
    load_eval=True,  # loads eval_stats.csv
    load_detailed=False,  # doesn't load .json
    load_models=True,  # loads .pt models
    pretrained_dict=pretrained_dict_sym,
)

In [ ]:
pretrained_lm_models['pretrained'][0][object_id]['patch'].pos

In [ ]:
object_id = 'orange'

graph = pretrained_lm_models['pretrained'][0][object_id]['patch']
use_for_hyp_init = graph.use_for_hyp_init

fig = plt.figure()
ax = fig.add_subplot(1, 1, 1, projection="3d")
colors = [
    "grey" if x is None else "limegreen" if x else "cyan"
    for x in use_for_hyp_init
]
pos = graph.pos
ax.scatter(
    pos[:, 1],
    pos[:, 0],
    pos[:, 2],
    color=colors,
    s=10,
    alpha=0.2,
)
ax.set_title(f"Symmetric locations for {object_id}")
format_axes(ax)
plt.show()

In [ ]:
# Load eval_stats from path
eval_stats_path_default = os.path.expanduser("~/tbp/results/monty/projects/evidence_eval_runs/logs/base_77obj_surf_agent_not_marked_up/eval_stats.csv")
eval_stats_default = pd.read_csv(eval_stats_path_default)

eval_stats_path_marked_up = os.path.expanduser("~/tbp/results/monty/projects/evidence_eval_runs/logs/base_77obj_surf_agent_load_symmetry_once_marked_nn20_all/eval_stats.csv")
eval_stats_marked_up = pd.read_csv(eval_stats_path_marked_up)


In [ ]:
# compare average values in columns: num_steps, symmetry_evidence, primary_performance, time

for col in ["num_steps", "symmetry_evidence", "time"]:
    print(f"Average {col} for default: {eval_stats_default[col].mean()}")
    print(f"Average {col} for marked_up: {eval_stats_marked_up[col].mean()}")
    print(f"Difference: {eval_stats_default[col].mean() - eval_stats_marked_up[col].mean()}")
    print("\n")

# get percent correct and percent correct mlh for default and marked_up

print(f"Percent correct for default: {eval_stats_default['primary_performance'].value_counts().get('correct', 0) / len(eval_stats_default) * 100}")
print(f"Percent correct for marked_up: {eval_stats_marked_up['primary_performance'].value_counts().get('correct', 0) / len(eval_stats_marked_up) * 100}")


